# Notebook 6: Bandits (explore vs exploit, in its purest form)

Step 6 of the ladder. A **bandit** is the simplest reinforcement-learning problem: **one state, several actions, random rewards.** No maze, no future, no discounting. What remains is the hardest question in the whole field in its cleanest form: **do you keep doing what looks best (exploit), or try something else to learn more (explore)?** Every `epsilon` you've seen so far is an answer to that question.

You need: the "new = old + step x (target - old)" update (Notebook 1), expectations and the square-root law (Notebook 5), the softmax policy and baseline (Notebooks 2 and 5). NumPy and matplotlib only; runs in about a minute.

### What you'll be able to say at the end
- Why **pure greedy fails**, and *how often*.
- How **epsilon-greedy, optimism, UCB, Thompson sampling and a softmax policy** each handle exploration, and what each knob does.
- That the **gradient bandit is REINFORCE** in its simplest form, and you'll test the baseline and entropy ideas from Notebook 5 on actual learning.
- Why **constant step sizes** (like Q-learning's `alpha`) beat sample averages when the world changes.
- A fair way to compare methods: many random problems, regret instead of raw reward, and honest caveats.

Same markers as before: `WHAT / WHY / SAME AS / BECOMES / WATCH` comments, **PREDICT** cells, a knob ledger, and a sandbox.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# WHAT: plot styling only (same quiet look as earlier notebooks). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

print("ready")

ready


# Part 1: The problem, and why one look is not enough

> **You are here:** a machinist is choosing among 5 candidate cutting settings (the "arms" of the bandit). Each time you cut a part with a setting, you get a quality score. The score is **noisy**: the same setting gives different scores on different parts. You want the best average score over many parts, but you don't know which setting is best.

Formally: arm `a` has a hidden **true mean** `q*(a)`. Pulling it returns `q*(a)` plus noise (standard deviation 1). You want to pull the arm with the highest true mean as often as possible.

**The temptation:** try each arm once, then always pick the one that scored best. Is that wise?

**PREDICT first.** Five arms with true means `0.8, 0.3, 0.0, -0.5, -1.0` (so the first is truly best, by half a point over the next) and noise of 1. After one pull of each arm, how often is the arm that *looks* best actually the truly best arm: 90% of the time, 60%, or 30%? Type a guess, then run.

In [2]:
# PREDICT: after one pull of each of 5 noisy arms, how often does the best-LOOKING arm equal the truly best arm?
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [3]:
true_means = np.array([0.8, 0.3, 0.0, -0.5, -1.0])
rng = np.random.default_rng(0)
trials = 100_000
one_pull_each = true_means + rng.normal(0, 1.0, (trials, 5))        # WHAT: pull every arm once, in 100,000 repeated experiments
looks_best = one_pull_each.argmax(axis=1)                            # WHAT: the arm that scored highest on its single pull
print(f"the arm that looks best after one pull each is truly best in {np.mean(looks_best == 0):.0%} of experiments")
print("how often each arm 'looks best':", " ".join(f"arm{a}: {np.mean(looks_best == a):.0%}" for a in range(5)))
print(f"giving up on all the others after one look commits you to a WORSE arm {np.mean(looks_best != 0):.0%} of the time")

the arm that looks best after one pull each is truly best in 48% of experiments
how often each arm 'looks best': arm0: 48% arm1: 25% arm2: 17% arm3: 7% arm4: 3%
giving up on all the others after one look commits you to a WORSE arm 52% of the time


**What to notice.** The truly best arm looks best in only **48%** of experiments, so more than half the time one look points you at the wrong arm (even arm 4, the worst, looks best 3% of the time). One sample per arm is far too noisy (Notebook 5: a single sample is almost all noise). So *exploiting too early* locks in a mistake. But exploring forever wastes pulls on arms you already know are bad. That tension is the whole subject.

**Regret** makes this measurable: after each pull, the regret is `q*(best arm) - q*(arm you pulled)`: what you gave up by not pulling the best arm. Total regret is the sum. A good method keeps it low.

# Part 2: Estimating each arm (the update you already know)

> **You are here:** to choose, you need a running estimate `Q(a)` of each arm's mean. Keep one number per arm and update it after each pull.

The **sample average** after `n` pulls is the mean of the rewards so far. You don't need to store all the rewards: `Q_new = Q_old + (1/n) * (reward - Q_old)`. That is **Notebook 1's update**, `new = old + step * (target - old)`, with step size `1/n`. Check that it gives exactly the mean:

In [4]:
r = np.random.default_rng(1).normal(0.5, 1.0, 200)
Q, n = 0.0, 0
for reward in r:
    n += 1
    Q = Q + (1 / n) * (reward - Q)            # WHAT: the incremental update, step size 1/n (big early, small later)
print(f"incremental estimate: {Q:.12f}")
print(f"np.mean of the data : {r.mean():.12f}")
print(f"difference          : {abs(Q - r.mean()):.2e}")
print()
print("with a CONSTANT step size alpha, old rewards fade instead of counting equally:")
alpha = 0.1
for k in (0, 5, 10, 20, 50):
    print(f"   weight on the reward from {k:2d} pulls ago: {alpha * (1 - alpha) ** k:.4f}   (the sample average would give 1/200 = 0.0050 to every reward)")

incremental estimate: 0.426337533605
np.mean of the data : 0.426337533605
difference          : 1.11e-16

with a CONSTANT step size alpha, old rewards fade instead of counting equally:
   weight on the reward from  0 pulls ago: 0.1000   (the sample average would give 1/200 = 0.0050 to every reward)
   weight on the reward from  5 pulls ago: 0.0590   (the sample average would give 1/200 = 0.0050 to every reward)
   weight on the reward from 10 pulls ago: 0.0349   (the sample average would give 1/200 = 0.0050 to every reward)
   weight on the reward from 20 pulls ago: 0.0122   (the sample average would give 1/200 = 0.0050 to every reward)
   weight on the reward from 50 pulls ago: 0.0005   (the sample average would give 1/200 = 0.0050 to every reward)


**What to notice.** The step size `1/n` makes the update an *exact* average where every reward counts the same. A **constant** `alpha` makes the estimate an average that **forgets**: recent rewards weigh more (a geometric fade, like the weights printed). Constant `alpha` is what Q-learning used. Part 9 shows why you'd want forgetting.

# Part 3: The testbed, and pure greedy

> **You are here:** a fair way to compare methods. One bandit problem is far too random to judge from. So:

- **500 different random problems**, each with **10 arms**. In each, the true arm means are drawn from a normal distribution (mean 0, spread 1), so the best arm differs between problems.
- Each run is **1000 pulls**. Report averages across the 500 problems.
- Every method sees the **identical** problems and the **identical** noise (we pre-generate it), so differences are the method's, not luck of the draw.

The code below builds that testbed and a small framework: an *agent* has `act(t)` (choose an arm for every problem at once) and `learn(a, r)`. Vectorized over the 500 problems, so it's fast.

In [5]:
P, K, T = 500, 10, 1000
IDX = np.arange(P)

def make_scenario(kind, seed):
    # WHAT: build 500 bandit problems. kind: "standard" (true means ~ N(0,1)), "shifted" (means ~ N(4,1)), "drifting" (means wander over time).
    r = np.random.default_rng(seed)
    base = r.normal(0, 1, (P, K))
    if kind == "shifted": base = base + 4.0
    noise = r.normal(0, 1, (P, K, T)).astype(np.float32)           # WHAT: the reward noise for every (problem, arm, pull), fixed in advance
    drift = np.cumsum(r.normal(0, 0.03, (P, K, T)), axis=2).astype(np.float32) if kind == "drifting" else None
    return dict(kind=kind, base=base, noise=noise, drift=drift)

SCEN = {k: make_scenario(k, 100 + i) for i, k in enumerate(("standard", "shifted", "drifting"))}

def run(make_agent, scen, seed=1234):
    agent = make_agent(np.random.default_rng(seed))
    base, noise, drift = scen["base"], scen["noise"], scen["drift"]
    regret, optimal, reward = np.zeros(T), np.zeros(T), np.zeros(T)
    for t in range(T):
        means = base if drift is None else base + drift[:, :, t]       # WHAT: this pull's true means (they move in the drifting scenario)
        a = agent.act(t)                                               # WHAT: the agent picks one arm per problem
        rew = means[IDX, a] + noise[IDX, a, t]                         # WHAT: the noisy reward it gets
        agent.learn(a, rew)
        best = means.max(axis=1)
        regret[t] = np.mean(best - means[IDX, a])                      # WHAT: regret = (best arm's true mean) - (true mean of the arm pulled)
        optimal[t] = np.mean(means[IDX, a] == best)
        reward[t] = rew.mean()
    return dict(regret=regret, optimal=optimal, reward=reward)

def smooth(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")              # WHAT: moving average so curves are readable (plots only)

def argmax_random_ties(x, rng):
    return (x + 1e-9 * rng.random(x.shape)).argmax(axis=1)           # WHAT: argmax that breaks ties at random (otherwise "first arm" wins every tie)

class EpsGreedy:
    # WHAT: with probability eps pull a random arm; otherwise pull the arm with the highest estimate so far.
    def __init__(self, rng, eps=0.1, alpha=None, q0=0.0):
        self.rng, self.eps, self.alpha = rng, eps, alpha
        self.Q = np.full((P, K), float(q0)); self.N = np.zeros((P, K))
    def act(self, t):
        greedy = argmax_random_ties(self.Q, self.rng)
        explore = self.rng.random(P) < self.eps
        return np.where(explore, self.rng.integers(0, K, P), greedy)
    def learn(self, a, r):
        self.N[IDX, a] += 1
        step = 1.0 / self.N[IDX, a] if self.alpha is None else self.alpha   # WHAT: step size: 1/n (exact average) or a constant alpha
        self.Q[IDX, a] += step * (r - self.Q[IDX, a])                       # SAME AS: Notebook 1's update (and the Qmaze update)

def summarize(res, last=100):
    return dict(reward=float(res["reward"].mean()), regret=float(res["regret"].mean()), optimal_end=float(res["optimal"][-last:].mean()))

greedy = run(lambda r: EpsGreedy(r, eps=0.0), SCEN["standard"])
print("pure greedy (eps = 0), 500 problems x 1000 pulls:")
print(f"   average reward per pull : {greedy['reward'].mean():.3f}")
print(f"   average regret per pull : {greedy['regret'].mean():.3f}")
print(f"   picks the best arm in the last 100 pulls: {greedy['optimal'][-100:].mean():.0%}")
print(f"   (for reference, the best arm's mean averages {SCEN['standard']['base'].max(1).mean():.3f}; a random arm averages 0)")

pure greedy (eps = 0), 500 problems x 1000 pulls:
   average reward per pull : 1.055
   average regret per pull : 0.487
   picks the best arm in the last 100 pulls: 38%
   (for reference, the best arm's mean averages 1.540; a random arm averages 0)


**What to notice.** Pure greedy gets an average reward of 1.06 per pull while 1.54 was available from the best arm: it gives up **0.49 per pull**. In the last 100 pulls it chooses the best arm in only **38%** of problems (guessing among 10 arms would give 10%). Its curve in the Part 4 figure goes completely **flat after about 100 pulls: it never changes its mind.** Whichever arm happened to pay well early keeps a high estimate and keeps getting pulled; the others are never pulled again, so their estimates never get a chance to improve.

Greedy isn't bad *at first* (its first 50 pulls average 0.94, as good as anything in the next table). It is bad *forever*.

# Part 4: Epsilon-greedy (the knob from every earlier notebook)

> **You are here:** the simplest fix: with probability `epsilon` pull a random arm, otherwise pull the best-looking one.

**PREDICT first.** Try `epsilon = 0` (never explore), `0.01`, `0.1`, `0.3`. After 1000 pulls, which will have the **highest average reward at the end**? Which will be **best in the first 50 pulls**? Type a guess, then run.

In [6]:
# PREDICT: which epsilon is best at the very end? which is best at the very start?
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [7]:
eps_list = [0.0, 0.01, 0.1, 0.3]
eps_runs = {e: run(lambda r, e=e: EpsGreedy(r, eps=e), SCEN["standard"]) for e in eps_list}
print(" epsilon | avg reward, first 50 pulls | avg reward, last 100 pulls | % best arm, last 100 | avg regret per pull")
for e in eps_list:
    rr = eps_runs[e]
    print(f"  {e:5} |          {rr['reward'][:50].mean():6.3f}            |          {rr['reward'][-100:].mean():6.3f}            |        {rr['optimal'][-100:].mean():5.0%}        |      {rr['regret'].mean():.3f}")

 epsilon | avg reward, first 50 pulls | avg reward, last 100 pulls | % best arm, last 100 | avg regret per pull
    0.0 |           0.939            |           1.066            |          38%        |      0.487
   0.01 |           0.938            |           1.329            |          62%        |      0.322
    0.1 |           0.903            |           1.372            |          80%        |      0.231
    0.3 |           0.772            |           1.077            |          67%        |      0.490


In [8]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.1))
cols = {0.0: YELLOW, 0.01: AQUA, 0.1: BLUE, 0.3: ORANGE}
for e in eps_list:
    y1, y2 = smooth(eps_runs[e]["reward"]), smooth(eps_runs[e]["optimal"]) * 100
    xs = np.arange(len(y1)) + 20
    a1.plot(xs, y1, color=cols[e], label=f"eps {e}"); a2.plot(xs, y2, color=cols[e], label=f"eps {e}")
    tag(a1, xs[-1], y1[-1], f"eps {e}", cols[e], dx=5, dy={0.3: 7, 0.0: -7}.get(e, 0)); tag(a2, xs[-1], y2[-1], f"eps {e}", cols[e], dx=5)
a1.set_xlabel("pull number"); a1.set_ylabel("average reward"); a1.set_title("Reward per pull (500 problems, smoothed)"); a1.set_xlim(0, 1130)
a2.set_xlabel("pull number"); a2.set_ylabel("% of problems pulling the best arm"); a2.set_title("How often the best arm is chosen"); a2.set_xlim(0, 1130)
plt.tight_layout(); plt.show()

<Figure size 1265x451 with 2 Axes>

**What to notice** (the answers to the PREDICT):
- **At the very start, `epsilon = 0` is best** (0.94 over the first 50 pulls, vs 0.90 for 0.1 and 0.77 for 0.3): exploration costs pulls you could have spent on the current favorite.
- **At the end, `epsilon = 0.1` is best** (reward 1.37, best arm chosen 80% of the time, regret 0.23 per pull). `0.01` is still climbing at pull 1000 (62% and rising): it explores so slowly that it hasn't finished learning. (It might catch up given many more pulls; I haven't run that.)
- **`epsilon = 0.3` hits a ceiling.** Even an agent that knew the best arm perfectly would pull it only 73% of the time (70% greedy, plus 3% by random luck); it reaches 67%. Constant heavy exploration is a permanent tax. Its final reward (1.08) ends up close to pure greedy's (1.07), for opposite reasons: greedy is stuck on the wrong arm, `0.3` wastes pulls on random arms.
- So **no single epsilon is right**: the best value depends on how many pulls you have left. That's why people shrink epsilon over time (a sandbox idea).

# Part 5: Optimism (explore by being hopeful)

> **You are here:** a different idea with *no random exploration at all*: start every estimate **too high** (say `Q = +5` when real rewards are around 0). The first pull on any arm disappoints, so the greedy rule moves on to an arm still sitting at the hopeful start. Greedy behaviour, but it tries everything.

For a fair comparison, give both methods the same constant step `alpha = 0.1` (Part 2).

In [9]:
opt_runs = {
    "greedy, Q0 = +5 (optimistic)": run(lambda r: EpsGreedy(r, eps=0.0, alpha=0.1, q0=5.0), SCEN["standard"]),
    "greedy, Q0 = 0 (realistic)":   run(lambda r: EpsGreedy(r, eps=0.0, alpha=0.1, q0=0.0), SCEN["standard"]),
    "eps 0.1, Q0 = 0":              run(lambda r: EpsGreedy(r, eps=0.1, alpha=0.1, q0=0.0), SCEN["standard"]),
}
print(" method                          | avg reward, first 50 | avg reward, last 100 | % best arm, last 100 | regret per pull")
for k, rr in opt_runs.items():
    print(f" {k:31s} |       {rr['reward'][:50].mean():6.3f}        |       {rr['reward'][-100:].mean():6.3f}        |        {rr['optimal'][-100:].mean():5.0%}        |     {rr['regret'].mean():.3f}")

 method                          | avg reward, first 50 | avg reward, last 100 | % best arm, last 100 | regret per pull
 greedy, Q0 = +5 (optimistic)    |        0.249        |        1.505        |          84%        |     0.241
 greedy, Q0 = 0 (realistic)      |        0.975        |        1.259        |          54%        |     0.326
 eps 0.1, Q0 = 0                 |        0.874        |        1.356        |          75%        |     0.292


**What to notice.**
- **Optimistic greedy** (no random exploration at all) picks the best arm **84%** of the time at the end, with regret 0.24 per pull. Epsilon-greedy with the same step size gets 75% and 0.29, and plain greedy with a realistic start gets 54% and 0.33.
- **The price is paid early:** over the first 50 pulls it averages just **0.25** reward per pull (the others get about 0.9), because it spends them trying arms that disappoint the inflated estimate. Optimism front-loads the exploration.
- The number `+5` is a guess about the reward scale: you need to know roughly how big rewards can be to pick it. (Part 10 shows it also did well in the shifted and drifting scenarios; I haven't investigated why in the shifted one, where `+5` is no longer above the best arm's true mean.)

# Part 6: UCB (explore where you're most uncertain)

> **You are here:** a smarter idea: not all uncertainty is equal. An arm pulled 3 times is poorly known; an arm pulled 300 times is well known. **UCB** (upper confidence bound) pulls the arm with the highest **estimate plus an uncertainty bonus**:

`score(a) = Q(a) + c * sqrt( ln(t) / N(a) )`

The bonus is big for rarely-pulled arms (small `N(a)`) and shrinks as an arm is pulled. `c` is the knob: how much to reward uncertainty. This is the square-root law of Notebook 5 turned into a decision rule: the bonus is a stand-in for "how big is the error in my average".

In [10]:
class UCB:
    def __init__(self, rng, c=1.0):
        self.rng, self.c = rng, c
        self.Q = np.zeros((P, K)); self.N = np.zeros((P, K))
    def act(self, t):
        bonus = self.c * np.sqrt(np.log(t + 1) / np.maximum(self.N, 1))                # WHAT: uncertainty bonus: larger when N is small
        score = np.where(self.N == 0, np.inf, self.Q + bonus)                          # WHAT: a never-tried arm gets an infinite score: try each once first
        return argmax_random_ties(score, self.rng)
    def learn(self, a, r):
        self.N[IDX, a] += 1
        self.Q[IDX, a] += (r - self.Q[IDX, a]) / self.N[IDX, a]

ucb_runs = {c: run(lambda r, c=c: UCB(r, c=c), SCEN["standard"]) for c in (0.1, 0.5, 1.0, 2.0)}
print("   c   | avg reward, first 50 | avg reward, last 100 | % best arm, last 100 | regret per pull")
for c, rr in ucb_runs.items():
    print(f" {c:4}  |       {rr['reward'][:50].mean():6.3f}        |       {rr['reward'][-100:].mean():6.3f}        |        {rr['optimal'][-100:].mean():5.0%}        |     {rr['regret'].mean():.3f}")

   c   | avg reward, first 50 | avg reward, last 100 | % best arm, last 100 | regret per pull
  0.1  |        1.084        |        1.455        |          73%        |     0.106
  0.5  |        1.079        |        1.518        |          85%        |     0.055
  1.0  |        1.015        |        1.529        |          92%        |     0.058
  2.0  |        0.767        |        1.493        |          85%        |     0.148


**What to notice.**
- UCB with `c` around 0.5 to 1 has **far lower regret (0.055 to 0.058) than the best epsilon-greedy (0.231)**, and chooses the best arm 85% to 92% of the time at the end.
- `c` has a sweet spot like epsilon did: `c = 0.1` explores too little (regret 0.106) and `c = 2` explores too much (regret 0.148, and a slow start: 0.77 reward over the first 50 pulls).
- Why it wins: epsilon-greedy keeps spending 10% of its pulls on *random* arms forever, including arms it is already sure are bad. UCB's bonus fades as an arm is pulled, so it stops wasting pulls on arms it knows.
- Fine print: I swept four values of `c` on this testbed and am showing the best, which flatters UCB a little. It also assumes a fixed world (sample averages) and rewards with noise near 1 (the bonus scale matters).

# Part 7: Thompson sampling (explore in proportion to your belief)

> **You are here:** a Bayesian idea. Keep a *belief* about each arm's mean (a bell curve: centered on your estimate, narrower as you pull more). To choose, **draw one random sample from each arm's belief and pull the arm whose sample is highest.** An arm you're unsure about has a wide belief, so it sometimes draws a high number and gets pulled. An arm you're sure is bad almost never does.

For this testbed (reward noise of 1, prior belief that arm means are around 0 with spread 1) the belief after `n` pulls with total reward `S` is a bell curve with mean `S / (n + 1)` and standard deviation `1 / sqrt(n + 1)`.

**Honest caveat:** this version is *told* the true noise level and the true spread of the arm means. The other methods aren't. That is a real advantage on the "standard" scenario; the "shifted" scenario (means near +4, not 0) breaks the prior's assumption, so you'll see what a wrong belief costs.

In [11]:
class Thompson:
    def __init__(self, rng, prior_mean=0.0):
        self.rng = rng; self.S = np.zeros((P, K)); self.N = np.zeros((P, K)); self.m0 = prior_mean
    def act(self, t):
        mean = (self.S + self.m0) / (self.N + 1)                                       # WHAT: belief centre: pulled reward total, plus a prior mean worth 1 pull
        sd = 1.0 / np.sqrt(self.N + 1)                                                 # WHAT: belief width: shrinks as N grows (square-root law again)
        return argmax_random_ties(mean + sd * self.rng.normal(size=(P, K)), self.rng)  # WHAT: draw one sample per arm; pull the highest draw
    def learn(self, a, r):
        self.N[IDX, a] += 1; self.S[IDX, a] += r

ts = run(lambda r: Thompson(r), SCEN["standard"])
print(f"Thompson (standard scenario): avg reward {ts['reward'].mean():.3f}, last-100 reward {ts['reward'][-100:].mean():.3f}, best arm in last 100: {ts['optimal'][-100:].mean():.0%}, regret/pull {ts['regret'].mean():.3f}")

Thompson (standard scenario): avg reward 1.469, last-100 reward 1.521, best arm in last 100: 90%, regret/pull 0.069


**What to notice.** Thompson sampling has **no exploration knob at all** and lands near UCB's best (regret 0.069, best arm 90% of the time, vs UCB's 0.058). Its exploration comes from the width of its beliefs.

But the price of "no knob" is that the knob moves into the **prior**: in Part 10's *shifted* scenario (true means near +4, prior says 0), this same agent's regret jumps to **0.90** and it picks the best arm only 19% of the time. With a prior of 0, arms it has never pulled are believed to be near 0, while arms it has pulled show rewards near 4, so the untried arms almost never win a draw: **a pessimistic prior is the opposite of optimism, and it kills exploration.** A well-matched prior makes Thompson sampling excellent; a wrong one makes it a trap.

# Part 8: The gradient bandit (this is REINFORCE)

> **You are here:** the last method uses no value estimates at all. It keeps a **preference** `H(a)` per arm and turns preferences into a policy with a softmax: `pi(a) = exp(H(a)) / sum( exp(H) )`. After each pull it nudges preferences with the **policy-gradient** update from Notebooks 2 and 5:

`H(a) += alpha * (reward - baseline) * ( 1[a was pulled] - pi(a) )`

**Same thing as REINFORCE**, with one state and one step. The baseline is the average reward so far. This lets us test Notebook 5's two claims on actual learning:
1. the **baseline** matters (and matters *more* when rewards are large), and
2. an **entropy bonus** `beta` keeps the policy from collapsing.

The "shifted" scenario (every arm's true mean raised by 4) is the test for the baseline: it changes no arm's *ranking*, only the size of the rewards, which is exactly what a baseline should absorb.

In [12]:
def softmax_rows(H):
    e = np.exp(H - H.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

class GradientBandit:
    def __init__(self, rng, alpha=0.1, use_baseline=True, beta=0.0):
        self.rng, self.alpha, self.use_b, self.beta = rng, alpha, use_baseline, beta
        self.H = np.zeros((P, K)); self.base = np.zeros(P); self.n = 0
    def act(self, t):
        self.pi = softmax_rows(self.H)                                                 # WHAT: preferences -> policy (a distribution over the 10 arms)
        u = self.rng.random(P)
        return np.minimum((np.cumsum(self.pi, axis=1) < u[:, None]).sum(axis=1), K - 1)   # WHAT: SAMPLE an arm from that distribution (Notebook 5, Part 1)
    def learn(self, a, r):
        onehot = np.zeros((P, K)); onehot[IDX, a] = 1.0
        b = self.base if self.use_b else 0.0                                           # WHAT: baseline = average of PAST rewards (never looks at this pull's action)
        grad = (r - b)[:, None] * (onehot - self.pi)                                   # WHAT: REINFORCE slope for a softmax policy (Notebook 2 Part 9, Notebook 5 Part 5)
        if self.beta > 0:
            ent = -(self.pi * np.log(self.pi + 1e-12)).sum(axis=1, keepdims=True)
            grad = grad + self.beta * (-self.pi * (np.log(self.pi + 1e-12) + ent))     # WHAT: ascend entropy too (slope checked in Notebook 5, Part 6)
        self.H += self.alpha * grad
        self.n += 1; self.base += (r - self.base) / self.n                             # WHAT: update the running average reward AFTER using it

gb_runs = {}
for scen_name in ("standard", "shifted"):
    for ub in (True, False):
        gb_runs[(scen_name, ub)] = run(lambda r, ub=ub: GradientBandit(r, alpha=0.1, use_baseline=ub), SCEN[scen_name])
print(" scenario  | baseline | avg reward, last 100 | % best arm, last 100 | regret per pull")
for (sn, ub), rr in gb_runs.items():
    print(f" {sn:9s} |   {'yes' if ub else 'no ':3s}    |       {rr['reward'][-100:].mean():6.3f}        |        {rr['optimal'][-100:].mean():5.0%}        |     {rr['regret'].mean():.3f}")

 scenario  | baseline | avg reward, last 100 | % best arm, last 100 | regret per pull
 standard  |   yes    |        1.505        |          84%        |     0.178
 standard  |   no     |        1.498        |          81%        |     0.185
 shifted   |   yes    |        5.469        |          81%        |     0.191
 shifted   |   no     |        5.169        |          45%        |     0.474


In [13]:
fig, ax = plt.subplots(figsize=(7.8, 4.1))
for (sn, ub), col, lab in (((("shifted", True)), BLUE, "with baseline"), ((("shifted", False)), ORANGE, "no baseline")):
    y = smooth(gb_runs[(sn, ub)]["optimal"]) * 100
    ax.plot(np.arange(len(y)) + 20, y, color=col)
    tag(ax, len(y) + 19, y[-1], lab, col, dx=5)
ax.set_xlim(0, 1180); ax.set_xlabel("pull number"); ax.set_ylabel("% of problems pulling the best arm")
ax.set_title("Rewards shifted up by 4: the baseline absorbs the shift")
plt.show()

<Figure size 858x451 with 1 Axes>

In [14]:
print("entropy bonus beta on the standard scenario (alpha 0.1, with baseline):")
print("  beta  | % best arm, last 100 | regret per pull | average policy entropy at the end (max = ln 10 = 2.303)")
beta_runs, beta_ent = {}, {}
for beta in (0.0, 0.01, 0.1, 1.0):
    ag_holder = {}
    def mk(r, beta=beta):
        ag = GradientBandit(r, alpha=0.1, use_baseline=True, beta=beta); ag_holder["a"] = ag; return ag
    rr = run(mk, SCEN["standard"]); beta_runs[beta] = rr
    pi_end = softmax_rows(ag_holder["a"].H)
    ent_end = float(np.mean(-(pi_end * np.log(pi_end + 1e-12)).sum(axis=1))); beta_ent[beta] = ent_end
    print(f" {beta:5} |        {rr['optimal'][-100:].mean():5.0%}        |     {rr['regret'].mean():.3f}     |        {ent_end:.3f}")

entropy bonus beta on the standard scenario (alpha 0.1, with baseline):
  beta  | % best arm, last 100 | regret per pull | average policy entropy at the end (max = ln 10 = 2.303)


   0.0 |          84%        |     0.178     |        0.114


  0.01 |          84%        |     0.180     |        0.130


   0.1 |          86%        |     0.195     |        0.275


   1.0 |          32%        |     0.786     |        1.891


**What to notice.**
- **Baseline (Notebook 5's claim, now on real learning).** On the standard scenario the baseline helps a little (best arm 84% vs 81%, regret 0.178 vs 0.185). With every reward shifted up by 4, **it is the difference between working and not**: 81% with the baseline vs 45% without (regret 0.191 vs 0.474). Without a baseline, every reward is large and positive, so every pull pushes up the preference of whatever arm was pulled. That is Notebook 5's "two big opposing shouts" problem, with the real signal buried inside.
- **Entropy bonus.** `beta = 0.01` changes almost nothing (best arm 84%, regret 0.180 vs 0.178). `beta = 0.1` gives a slightly higher best-arm rate (86%) but slightly higher regret (0.195): no clear benefit. `beta = 1.0` **ruins it** (32%, regret 0.79, entropy 1.89 out of a maximum 2.30, nearly random). So the knob has a cliff: small is harmless, large is fatal. **I could not show a benefit from the entropy bonus on this problem**; it's an insurance against collapse, and the policy here didn't collapse in a harmful way. (A2C's 0.01 sits on the harmless side. Whether it helps on CartPole is not tested here.)
- Note the policy still ends nearly deterministic with `beta = 0` (end entropy 0.11) while being right 84% of the time. Learning *should* sharpen the policy.
- The gradient bandit is *not* the best bandit method here (regret 0.178 vs UCB's 0.058). Policy gradient pays for its generality: it will work when there is a state, a future, and a neural network, which UCB and Thompson sampling in this form won't.

# Part 9: When the world changes (why constant alpha exists)

> **You are here:** in a real shop the best setting drifts (tool wear, material batches, temperature). In the **drifting** scenario each arm's true mean wanders as a random walk. Sample averages (equal weight forever) vs constant `alpha` (forgetting):

In [15]:
drift_runs = {
    "sample average (step 1/n)": run(lambda r: EpsGreedy(r, eps=0.1, alpha=None), SCEN["drifting"]),
    "constant alpha = 0.1":      run(lambda r: EpsGreedy(r, eps=0.1, alpha=0.1), SCEN["drifting"]),
    "constant alpha = 0.3":      run(lambda r: EpsGreedy(r, eps=0.1, alpha=0.3), SCEN["drifting"]),
}
static_check = {
    "sample average (step 1/n)": run(lambda r: EpsGreedy(r, eps=0.1, alpha=None), SCEN["standard"]),
    "constant alpha = 0.1":      run(lambda r: EpsGreedy(r, eps=0.1, alpha=0.1), SCEN["standard"]),
    "constant alpha = 0.3":      run(lambda r: EpsGreedy(r, eps=0.1, alpha=0.3), SCEN["standard"]),
}
print(" eps 0.1 in both            | DRIFTING: regret per pull (last 500) | STATIC: regret per pull (last 500)")
for k in drift_runs:
    print(f" {k:26s} |               {drift_runs[k]['regret'][-500:].mean():.3f}                  |              {static_check[k]['regret'][-500:].mean():.3f}")

 eps 0.1 in both            | DRIFTING: regret per pull (last 500) | STATIC: regret per pull (last 500)
 sample average (step 1/n)  |               0.373                  |              0.177
 constant alpha = 0.1       |               0.330                  |              0.210
 constant alpha = 0.3       |               0.275                  |              0.236


In [16]:
fig, ax = plt.subplots(figsize=(7.8, 4.1))
for (k, rr), col in zip(drift_runs.items(), (ORANGE, BLUE, AQUA)):
    y = smooth(rr["regret"], 40)
    ax.plot(np.arange(len(y)) + 40, y, color=col)
    tag(ax, len(y) + 39, y[-1], k.replace("constant ", ""), col, dx=5)
ax.set_xlim(0, 1260); ax.set_xlabel("pull number"); ax.set_ylabel("regret per pull (lower is better)")
ax.set_title("Arms drift: methods that never forget fall behind")
plt.show()

<Figure size 858x451 with 1 Axes>

**What to notice.**
- In the figure, the sample-average method (orange) is competitive for the first several hundred pulls, then its regret **climbs** after about pull 600 as arms drift away from the long-ago rewards it still weighs equally, while the constant-step methods stay flat. Over the last 500 pulls: sample average 0.373, `alpha = 0.1` 0.330, `alpha = 0.3` 0.275.
- On the **static** scenario the order reverses (sample average 0.177, `alpha = 0.1` 0.210, `alpha = 0.3` 0.236): constant `alpha` never settles, because each new noisy reward moves the estimate by a fixed fraction. **Constant alpha trades accuracy on a stable world for the ability to track a changing one.** It's a dial between "trust all history" and "trust recent data".
- This is the `alpha` in your Q-learning code. (Deep RL has a second reason, not tested here: the thing being learned, the Q-targets, itself keeps changing as the network changes.)

# Part 10: Sandbox (your ideas compete here)

> **You are here:** all the methods are now on one ruler. Build your own agent and race it.

**The ruler.** An agent is a class with `act(t)` (return one arm index per problem, an array of length 500) and `learn(a, r)` (get the arms pulled and the rewards). It's scored in **three scenarios** by **average regret per pull** (lower is better) and **% best arm in the last 200 pulls**: *standard* (arm means around 0), *shifted* (arm means around 4, same rankings), and *drifting* (the best arm changes). Reference entries use settings from Parts 4 to 9 (a few I picked by looking at the results above, so they are mildly favored; yours should be tuned fairly: on the standard scenario only, and then see what happens in the other two).

In [17]:
def race6(entries):
    names = list(SCEN)
    print(f"{'':34s}" + "".join(f"{n:>26s}" for n in names))
    print(f"{'':34s}" + "".join(f"{'regret/pull | % best, end':>26s}" for _ in names))
    out = {}
    for label, factory in entries:
        row = []
        for n in names:
            rr = run(factory, SCEN[n])
            row.append((float(rr["regret"].mean()), float(rr["optimal"][-200:].mean())))
        out[label] = row
        print(f"{label:34s}" + "".join(f"{f'{a:.3f}  |  {b:.0%}':>26s}" for a, b in row))
    return out

reference6 = [
    ("eps-greedy 0.1 (sample avg)",      lambda r: EpsGreedy(r, eps=0.1)),
    ("eps-greedy 0.1, alpha 0.1",        lambda r: EpsGreedy(r, eps=0.1, alpha=0.1)),
    ("greedy, optimistic Q0 = +5",       lambda r: EpsGreedy(r, eps=0.0, alpha=0.1, q0=5.0)),
    ("UCB c = 1",                        lambda r: UCB(r, c=1.0)),
    ("Thompson (prior mean 0)",          lambda r: Thompson(r)),
    ("gradient bandit, baseline",        lambda r: GradientBandit(r, alpha=0.1)),
]
race_out = race6(reference6)

                                                    standard                   shifted                  drifting
                                   regret/pull | % best, end regret/pull | % best, end regret/pull | % best, end


eps-greedy 0.1 (sample avg)                    0.231  |  79%             0.263  |  77%             0.364  |  58%


eps-greedy 0.1, alpha 0.1                      0.292  |  74%             1.093  |  33%             0.375  |  67%


greedy, optimistic Q0 = +5                     0.241  |  84%             0.083  |  83%             0.274  |  75%


UCB c = 1                                      0.058  |  91%             0.059  |  91%             0.148  |  69%


Thompson (prior mean 0)                        0.069  |  90%             0.902  |  19%             0.178  |  65%


gradient bandit, baseline                      0.178  |  84%             0.191  |  81%             0.342  |  56%


**What to notice.**
- **UCB (c = 1) has the lowest regret in all three scenarios** (0.058, 0.059, 0.148). Thompson is close on the standard one (0.069) but collapses on the shifted one (0.90). Epsilon-greedy with `alpha = 0.1` and `Q0 = 0` also collapses on the shifted scenario (1.09, best arm 33%) for the *same* reason: estimates start at 0 while real rewards are near 4, so any arm tried once looks wonderful next to the untried ones.
- **Optimistic greedy** is the next best on shifted (0.083) and does well on drifting (0.274).
- **I expected a forgetting method to win the drifting scenario.** UCB with plain averages did best anyway (0.148, vs 0.274 for the optimistic constant-step agent), and I haven't investigated why.
- Caveats: some reference settings (`c`, `alpha`, `Q0`) were picked by looking at the results above; there is one set of 500 problems per scenario; and differences of about 0.01 in regret shouldn't be trusted. One-line summary: *methods that explore in proportion to uncertainty beat methods that explore at random, but methods that assume a fixed scale (a prior, an initial value) break when the scale changes.*

### Your slot
Edit `MyAgent` and re-run. Ideas: epsilon that **shrinks over time** (`eps = 1/sqrt(t)`); UCB with a constant step size so it can track drift; Thompson with a prior mean that learns from the data; an agent that switches strategy when it notices rewards drifting; or something of your own (even something silly: how much worse is it?). Rules: use only what `learn` has told you; no peeking at the true means.

In [18]:
class MyAgent:
    def __init__(self, rng):
        self.rng = rng; self.Q = np.zeros((P, K)); self.N = np.zeros((P, K))
    def act(self, t):
        # SANDBOX: replace this with your own idea. Return an integer array of length P (one arm per problem).
        eps = 0.1
        greedy = argmax_random_ties(self.Q, self.rng)
        return np.where(self.rng.random(P) < eps, self.rng.integers(0, K, P), greedy)
    def learn(self, a, r):
        self.N[IDX, a] += 1
        self.Q[IDX, a] += (r - self.Q[IDX, a]) / self.N[IDX, a]       # sample average; try a constant alpha, or something smarter

_ = race6(reference6 + [("MY AGENT", lambda r: MyAgent(r))])

                                                    standard                   shifted                  drifting
                                   regret/pull | % best, end regret/pull | % best, end regret/pull | % best, end


eps-greedy 0.1 (sample avg)                    0.231  |  79%             0.263  |  77%             0.364  |  58%


eps-greedy 0.1, alpha 0.1                      0.292  |  74%             1.093  |  33%             0.375  |  67%


greedy, optimistic Q0 = +5                     0.241  |  84%             0.083  |  83%             0.274  |  75%


UCB c = 1                                      0.058  |  91%             0.059  |  91%             0.148  |  69%


Thompson (prior mean 0)                        0.069  |  90%             0.902  |  19%             0.178  |  65%


gradient bandit, baseline                      0.178  |  84%             0.191  |  81%             0.342  |  56%


MY AGENT                                       0.231  |  79%             0.263  |  77%             0.364  |  58%


# Part 11: The knob ledger, self-check, and what's next

## Knob ledger (Notebooks 1 to 6)
Families: **objective**, **model**, **search**, **experience**, **judging**. New in this notebook: the exploration knobs.

In [19]:
g = summarize(greedy); e1 = summarize(eps_runs[0.1]); e3 = summarize(eps_runs[0.3])
rows = [
    ("learning rate / step size (NB1)", "search", "too small: slow", "too big: blows up (here: constant alpha forgets, 1/n never does)"),
    ("number of samples (NB5)", "experience", "few: noisy estimates", "100x more: 10x less noise (square-root law)"),
    ("epsilon", "search (exploration)", f"0: regret {g['regret']:.2f}/pull, best arm {g['optimal_end']:.0%} at the end", f"0.3: regret {e3['regret']:.2f}/pull, best arm {e3['optimal_end']:.0%} at the end  (0.1: {e1['regret']:.2f}, {e1['optimal_end']:.0%})"),
    ("initial value Q0", "search (exploration)", "0: greedy can lock in early", "+5 (optimistic): explores early via disappointment; costly in the first 50 pulls"),
    ("UCB exploration weight c", "search (exploration)", f"c = 0.1: regret {ucb_runs[0.1]['regret'].mean():.2f}", f"c = 2: regret {ucb_runs[2.0]['regret'].mean():.2f}  (c = 1: {ucb_runs[1.0]['regret'].mean():.2f})"),
    ("prior belief (Thompson)", "model", "matched to the world: strong", "wrong prior: regret 0.90 in the shifted scenario (Part 10)"),
    ("baseline (policy gradient)", "objective", f"none, shifted rewards: regret {gb_runs[('shifted', False)]['regret'].mean():.2f}", f"with baseline: regret {gb_runs[('shifted', True)]['regret'].mean():.2f}"),
    ("entropy weight beta", "objective", f"0: regret {beta_runs[0.0]['regret'].mean():.2f}, end entropy {beta_ent[0.0]:.2f}", f"1.0: regret {beta_runs[1.0]['regret'].mean():.2f}, end entropy {beta_ent[1.0]:.2f} (0.01: {beta_runs[0.01]['regret'].mean():.2f})"),
    ("constant vs 1/n step", "search", f"1/n on drifting arms: regret {drift_runs['sample average (step 1/n)']['regret'][-500:].mean():.2f}", f"alpha 0.1: {drift_runs['constant alpha = 0.1']['regret'][-500:].mean():.2f}"),
]
print(f"{'knob':34s}{'family':22s}{'at one extreme':62s}{'at the other'}")
print("-" * 200)
for knob, fam, a, b in rows:
    print(f"{knob:34s}{fam:22s}{a:62s}{b}")

knob                              family                at one extreme                                                at the other
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
learning rate / step size (NB1)   search                too small: slow                                               too big: blows up (here: constant alpha forgets, 1/n never does)
number of samples (NB5)           experience            few: noisy estimates                                          100x more: 10x less noise (square-root law)
epsilon                           search (exploration)  0: regret 0.49/pull, best arm 38% at the end                  0.3: regret 0.49/pull, best arm 67% at the end  (0.1: 0.23, 80%)
initial value Q0                  search (exploration)  0: greedy can lock in early                                   +5 (optimistic): explo

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| Why does pure greedy fail? | Early estimates are noisy; it locks onto whichever arm got lucky first and never checks the others. |
| What does epsilon trade off? | Exploring (learning, but pulling worse arms) against exploiting (best known arm, but possibly the wrong one). |
| What does optimism do? | A too-high start makes every first pull a disappointment, so a greedy rule tries everything once. The exploration is front-loaded, and it costs reward early (0.25 per pull over the first 50 pulls vs about 0.9 for epsilon-greedy). |
| How is UCB different from epsilon-greedy? | It explores *where uncertainty is largest* rather than at random; the bonus is the square-root law as a decision rule. |
| What does Thompson sampling do? | Draws one sample from each arm's belief and pulls the highest. Uncertain arms win some draws. It needs a reasonable prior. |
| Why is the gradient bandit REINFORCE? | A softmax policy, nudged by (reward minus baseline) x (chosen minus probability), is exactly the policy-gradient update. |
| Why do Q-learning and DQN use a constant alpha? | It forgets old data, which tracks a changing world (and a changing target), at the cost of never fully settling. |
| Why compare on regret over many problems? | One problem is mostly luck; regret removes the effect of the reward's size and measures what you gave up. |

## What's next
- You've seen every exploration idea RL uses, in miniature. Epsilon-greedy and entropy bonuses are in your CartPole and Qmaze code.
- The gradient bandit is REINFORCE with the maze removed. What's missing is the **state** and the **future**.

**Notebook 7: MDPs and value iteration.** Add states and consequences: actions now change where you are, and rewards come later. You'll write the Bellman equation as code, solve small worlds exactly with value iteration (including your Qmaze), and see Q-learning as the *sampled* version of the same equation.